In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm
import torch
from sklearn.impute import SimpleImputer

os.environ["TABPFN_TOKEN"] = "YOUR_TABPFN_TOKEN_HERE"

CACHE_DIR = "./model_weights_cache"
os.makedirs(CACHE_DIR, exist_ok=True)
os.environ["HF_HOME"] = f"{CACHE_DIR}/huggingface"
os.environ["TABPFN_MODEL_CACHE_DIR"] = f"{CACHE_DIR}/tabpfn"
os.environ["TABFM_MODEL_CACHE_DIR"] = f"{CACHE_DIR}/tabfm"

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

HF_TOKEN = "YOUR_HUGGINGFACE_TOKEN_HERE"
if HF_TOKEN: os.environ["HF_TOKEN"] = HF_TOKEN

CFG = {
    "DATA_PATH": "YOUR_DATASET_FILE.fthr",
    "TIME_COL": "date",
    "ENTITY_COL": "permno",
    "RETURN_COL": "return",
    "FEATURE_COLS": None,
    "HORIZON": 1,
    "BUFFER": 0,
    "TRAIN_WINDOW": 60,
    "MIN_ENTITIES": 50,
    "MAX_TRAIN_ROWS": 1000,
    "PERIODS_PER_YEAR": 12,
    "EXPERIMENT_LABEL": "TabFM_vs_TabPFN_NoImputation",
    "LOG_PATH": "YOUR_OUTPUT_LOG.csv",
    "FLUSH_EVERY": 6,
}

def load_panel(path):
    ext = path.lower().rsplit(".", 1)[-1]
    if ext == "csv": return pd.read_csv(path)
    if ext == "parquet": return pd.read_parquet(path)
    if ext in ("feather", "fthr"): return pd.read_feather(path)
    raise ValueError(f"Unrecognised extension '.{ext}' for {path}")

def gaussian_rank_weights(pct_ranks, clip=3.0):
    z = np.clip(norm.ppf(np.clip(pct_ranks, 1e-4, 1 - 1e-4)), -clip, clip)
    denom = np.abs(z).sum()
    return z / denom if denom > 0 else np.zeros_like(z)

def turnover(w_new, entities_new, w_old, entities_old):
    if w_old is None: return 1.0
    s_old = pd.Series(w_old, index=entities_old)
    s_new = pd.Series(w_new, index=entities_new)
    aligned = pd.concat([s_old, s_new], axis=1).fillna(0.0)
    return 0.5 * (aligned.iloc[:, 1] - aligned.iloc[:, 0]).abs().sum()

class TabPFNWrapper:
    def __init__(self, random_state=42):
        self.random_state = random_state
        self._imputer = SimpleImputer(strategy='constant', fill_value=0.0, add_indicator=True)

        print("Loading TabPFN weights into GPU memory...")
        from tabpfn import TabPFNRegressor
        self._m = TabPFNRegressor(
            device=DEVICE,
            n_estimators=1,
            ignore_pretraining_limits=True,
            random_state=self.random_state,
        )

    def fit(self, X, y):
        X_encoded = self._imputer.fit_transform(X)
        self._m.fit(X_encoded, y)
        return self

    def predict(self, X):
        X_encoded = self._imputer.transform(X)
        try:
            return self._m.predict(X_encoded, output_type="mean")
        except TypeError:
            return self._m.predict(X_encoded)

class TabFMWrapper:
    def __init__(self, random_state=42):
        self.random_state = random_state
        self._imputer = SimpleImputer(strategy='constant', fill_value=0.0, add_indicator=True)

        print("Loading TabFM weights into GPU memory...")
        from tabfm import TabFMRegressor
        from tabfm import tabfm_v1_0_0_pytorch as tabfm_v1

        base_model = tabfm_v1.load(model_type='regression')

        if DEVICE == "cuda":
            base_model = base_model.to("cuda")
        self._m = TabFMRegressor(model=base_model)

    def fit(self, X, y):
        X_encoded = self._imputer.fit_transform(X)
        self._m.fit(X_encoded, y)
        return self

    def predict(self, X):
        X_encoded = self._imputer.transform(X)
        return self._m.predict(X_encoded)

def build_model(name):
    if name == "TabPFN": return TabPFNWrapper()
    if name == "TabFM": return TabFMWrapper()
    raise ValueError(f"No builder for model '{name}'")

class WalkForwardBacktester:
    def __init__(self, df, cfg):
        self.cfg = cfg
        self.time_col = cfg["TIME_COL"]
        self.entity_col = cfg["ENTITY_COL"]
        self.return_col = cfg["RETURN_COL"]
        self.feature_cols = cfg.get("FEATURE_COLS")
        self.df = df.copy()
        self._prepare_data()

    def _prepare_data(self):
        df, cfg, K = self.df, self.cfg, self.cfg["HORIZON"] + self.cfg["BUFFER"]
        times = sorted(df[self.time_col].unique())
        pos = {t: i for i, t in enumerate(times)}
        fwd_of = {t: times[i + K] for t, i in pos.items() if i + K < len(times)}
        df["_fwd_time_key"] = df[self.time_col].map(fwd_of)

        fwd_returns = df[[self.entity_col, self.time_col, self.return_col]].rename(
            columns={self.time_col: "_fwd_time_key", self.return_col: "_fwd_ret"})
        df = df.merge(fwd_returns, on=[self.entity_col, "_fwd_time_key"], how="left")

        df["_target_rank"] = df.groupby("_fwd_time_key")["_fwd_ret"].rank(pct=True)

        if self.feature_cols is None:
            reserved = {self.time_col, self.entity_col, self.return_col,
                        "_fwd_time_key", "_fwd_ret", "_target_rank"}
            self.feature_cols = [c for c in df.columns if c not in reserved]

        self.df = df.dropna(subset=["_target_rank"]).reset_index(drop=True)
        self.times = sorted(self.df[self.time_col].unique())

    def run(self, models_to_run, log_path=None):
        cfg, df = self.cfg, self.df
        log_path = log_path or cfg["LOG_PATH"]
        times, W = self.times, cfg["TRAIN_WINDOW"]
        min_entities = cfg.get("MIN_ENTITIES", 30)

        X_ALL = df[self.feature_cols].values.astype(float)
        Y_RANK = df["_target_rank"].values.astype(float)
        Y_RET = df["_fwd_ret"].values.astype(float)
        ENT_ALL = df[self.entity_col].values
        time_to_rows = df.groupby(self.time_col).indices

        fields = ["ym", "model", "ic", "turnover", "gross_ret", "n_stocks", "resumed"]
        if cfg.get("EXPERIMENT_LABEL"): fields.append("experiment")

        done = set()
        if os.path.exists(log_path):
            prior = pd.read_csv(log_path)
            done = set(zip(prior["ym"].astype(str), prior["model"]))

        print("\nInitializing models (this will take a moment...)")
        active_models = {}
        for m in models_to_run:
            active_models[m] = build_model(m)
        print("Models initialized successfully. Starting loop...\n")

        rows, prev_weights = [], {}
        n_folds = len(times) - W
        t0 = time.time()

        for i, t in enumerate(times):
            if i < W: continue
            window_times = times[max(0, i - W):i]
            tr_idx = (np.concatenate([time_to_rows[wt] for wt in window_times]) if window_times else np.array([], dtype=int))
            te_idx = time_to_rows[t]
            if len(te_idx) < min_entities or len(tr_idx) < min_entities: continue

            Xtr, ytr = X_ALL[tr_idx], Y_RANK[tr_idx]
            Xte = X_ALL[te_idx]

            if len(Xtr) > cfg["MAX_TRAIN_ROWS"]:
                rng = np.random.RandomState(42 + i)
                sub = rng.choice(len(Xtr), cfg["MAX_TRAIN_ROWS"], replace=False)
                Xtr, ytr = Xtr[sub], ytr[sub]

            if i % 5 == 0:
                elapsed = time.time() - t0
                print(f"Fold {i - W + 1}/{n_folds} -- Period {t} -- {elapsed:.0f}s elapsed")

            for model_name in models_to_run:
                if (str(t), model_name) in done: continue
                was_cold_start = model_name not in prev_weights

                model = active_models[model_name]
                model.fit(Xtr, ytr)
                preds = model.predict(Xte)

                ic = pd.Series(preds).corr(pd.Series(Y_RANK[te_idx]), method="spearman")
                w = gaussian_rank_weights(pd.Series(preds).rank(pct=True).values)
                ents = ENT_ALL[te_idx]
                to = turnover(w, ents, *prev_weights.get(model_name, (None, None)))
                prev_weights[model_name] = (w, ents)
                gross = float(np.nansum(w * Y_RET[te_idx]))

                row = {"ym": t, "model": model_name, "ic": ic, "turnover": to,
                       "gross_ret": gross, "n_stocks": len(te_idx), "resumed": int(was_cold_start)}
                if cfg.get("EXPERIMENT_LABEL"): row["experiment"] = cfg["EXPERIMENT_LABEL"]
                rows.append(row)

            if rows and i % cfg.get("FLUSH_EVERY", 12) == 0:
                self._flush(rows, log_path, fields)
                rows = []

        if rows: self._flush(rows, log_path, fields)
        print(f"Done -- results saved to {log_path}")

    def _flush(self, rows, log_path, fields):
        new_df = pd.DataFrame(rows)[fields]
        header = not os.path.exists(log_path)
        new_df.to_csv(log_path, mode="a", header=header, index=False)

def generate_quant_tear_sheet(file_path, periods_per_year=12):
    print("\n✅ Loading and calculating performance metrics...")
    df = pd.read_csv(file_path)
    df.columns = df.columns.str.lower()

    df['ym_clean'] = df['ym'].astype(str).str.split('/').str[0].str.replace(r'\.0$', '', regex=True)
    df['ym'] = pd.to_datetime(df['ym_clean'], format='mixed')
    df = df.sort_values('ym')

    target_models = ['TabPFN', 'TabFM']
    df = df[df['model'].isin(target_models)]

    returns_df = df.pivot_table(index='ym', columns='model', values='gross_ret')
    cum_returns = (1 + returns_df).cumprod() - 1

    roll_max = (1 + returns_df).cumprod().cummax()
    drawdowns = ((1 + returns_df).cumprod() / roll_max) - 1

    rolling_sharpe = (
        returns_df.rolling(window=12).mean() / returns_df.rolling(window=12).std()
    ) * np.sqrt(periods_per_year)

    summary_stats = []
    for model in target_models:
        if model not in returns_df.columns:
            continue

        model_ret = returns_df[model]
        ann_ret = model_ret.mean() * periods_per_year
        ann_vol = model_ret.std() * np.sqrt(periods_per_year)
        sharpe = ann_ret / ann_vol if ann_vol != 0 else 0
        max_dd = drawdowns[model].min()

        avg_ic = df[df['model'] == model]['ic'].mean()
        avg_turnover = df[df['model'] == model]['turnover'].mean()

        summary_stats.append({
            'Model': model,
            'Ann. Return': f"{ann_ret:.2%}",
            'Ann. Volatility': f"{ann_vol:.2%}",
            'Sharpe Ratio': f"{sharpe:.2f}",
            'Max Drawdown': f"{max_dd:.2%}",
            'Mean IC': f"{avg_ic:.4f}",
            'Mean Turnover': f"{avg_turnover:.2%}"
        })

    summary_df = pd.DataFrame(summary_stats)

    print("\n" + "="*80)
    print("TABFM VS TABPFN - PERFORMANCE SUMMARY")
    print("="*80)
    print(summary_df.to_string(index=False))
    print("="*80 + "\n")

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle('TabFM vs TabPFN: Quantitative Tear Sheet', fontsize=16, fontweight='bold')
    colors = {'TabFM': '#1f77b4', 'TabPFN': '#ff7f0e'}

    for model in returns_df.columns:
        axes[0, 0].plot(cum_returns.index, cum_returns[model], label=model, color=colors.get(model), linewidth=2)
    axes[0, 0].set_title('Cumulative Returns')
    axes[0, 0].set_ylabel('Cumulative Return')
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].legend()
    axes[0, 0].axhline(0, color='black', linewidth=0.8, linestyle='--')

    for model in returns_df.columns:
        axes[0, 1].fill_between(drawdowns.index, drawdowns[model], 0, label=model, color=colors.get(model), alpha=0.3)
        axes[0, 1].plot(drawdowns.index, drawdowns[model], color=colors.get(model), linewidth=1)
    axes[0, 1].set_title('Underwater Drawdown')
    axes[0, 1].set_ylabel('Drawdown')
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].legend()

    for model in rolling_sharpe.columns:
        axes[1, 0].plot(rolling_sharpe.index, rolling_sharpe[model], label=model, color=colors.get(model), linewidth=1.5)
    axes[1, 0].set_title('Rolling 12-Period Sharpe Ratio')
    axes[1, 0].set_ylabel('Sharpe Ratio')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].axhline(0, color='black', linewidth=0.8, linestyle='--')
    axes[1, 0].legend()

    for model in returns_df.columns:
        model_data = df[df['model'] == model]
        axes[1, 1].scatter(model_data['turnover'], model_data['ic'], label=model, color=colors.get(model), alpha=0.6, edgecolors='w', s=50)
    axes[1, 1].set_title('Efficiency: IC vs Turnover')
    axes[1, 1].set_xlabel('Monthly Turnover')
    axes[1, 1].set_ylabel('Information Coefficient (IC)')
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].legend()
    axes[1, 1].axhline(0, color='black', linewidth=0.8, linestyle='--')

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

if __name__ == "__main__":
    print(f"Loading data from {CFG['DATA_PATH']}...")
    try:
        df = load_panel(CFG["DATA_PATH"])
        bt = WalkForwardBacktester(df, CFG)

        models_to_run = ["TabPFN", "TabFM"]

        print("Starting Backtest...")
        bt.run(models_to_run)

        generate_quant_tear_sheet(CFG["LOG_PATH"], periods_per_year=CFG["PERIODS_PER_YEAR"])
    except Exception as e:
        print(f"Execution Error: {e}")